### Set Up

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import string
import inflect
import pickle

%matplotlib inline

In [ ]:
import urllib
from urllib.request import urlopen
from bs4 import BeautifulSoup

In [ ]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
from nltk.stem import PorterStemmer
ps = PorterStemmer()

from string import punctuation
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

from textblob import TextBlob

In [ ]:
!pip install PyDictionary

from PyDictionary import PyDictionary

dictionary = PyDictionary()

# https://pypi.org/project/PyDictionary/

In [ ]:
# NLTK Definitions
# Ref: https://pythonprogramming.net/part-of-speech-tagging-nltk-tutorial/

pos_dict = {'CC':	'Coordinating Conjunction',
            'CD' : 'Cardinal Digit',
            'DT':	'Determiner',
            'EX' :	'existential there',
            'FW' :	'foreign word',
            'IN' : 'Preposition',
            'JJ' : 'Adjective',
            'JJR' : 'Adjective',
            'JJS' : 'Adjective',
            'MD':	'Modal',
            'NN' : 'Noun',
            'NNS': 'Noun',
            'NNP': 'Noun',
            'NNPS': 'Noun',
            'PDT' :	'Predeterminer',
            'POS':	'Possessive',
            'PRP' :	'Pronoun',
            'PRP$':	'Pronoun',
            'RB' : 'Adverb',
            'RBR' :	'Adverb',
            'RBS' :	'Adverb',
            'RP' :	'Particle',
            'TO':	'To',
            'UH':	'Interjection',
            'VB' : 'Verb',
            'VBD': 'Verb',
            'VBG': 'Verb',
            'VBN': 'Verb',
            'VBP': 'Verb',
            'VBZ': 'Verb',
            'WDT' :	'wh-determiner',
            'WP' : 'wh-pronoun',
            'WP$' :	'possessive wh-pronoun',
            'WRB' :	'wh-abverb'
}

# In all my experience with collecting new vocab, I never find an difficult word whose POS is not among {'Adjective', 'Adverb', 'Noun', 'Verb'}.

### Fetching Data from Internet and Saving it

In [ ]:
def get_book(book_url, num_chapters):

    ebook = []

    for i in range(0, num_chapters+1):

        chapter_url = book_url+str("{:02d}".format(i))+".html"
        print(chapter_url)
        html = urlopen(chapter_url).read()
        soup = BeautifulSoup(html)

        for script in soup(["script", "style"]):
            script.extract()

        ebook.append(soup.get_text())

    return ebook

In [ ]:
hhgtg = get_book("http://www.angelfire.com/ca3/tomsnyder/hg-1-", 35)
rateou = get_book("http://www.angelfire.com/ca3/tomsnyder/hg-2-", 34)
luae = get_book("http://www.angelfire.com/ca3/tomsnyder/hg-3-", 34)
slatfat = get_book("http://www.angelfire.com/ca3/tomsnyder/hg-4-", 41)
mh = get_book("http://www.angelfire.com/ca3/tomsnyder/hg-5-", 25)

In [ ]:
dont_panic = [hhgtg, rateou, luae, slatfat, mh]

In [ ]:
import pickle

pickle_out = open("dont_panic.pickle", "wb")
pickle.dump(dont_panic, pickle_out)
pickle_out.close()

### Dictionary

In [ ]:
word_pos_tags = []

for s in lem_word_tokens:
    wordsList = [w for w in s if w not in stop_words]
    word_pos_tags.append([TextBlob(wrd).tags for wrd in wordsList])

In [ ]:
words = []
pos = []

for s in word_pos_tags:
    for l in s:
        for w, p in l:
            words.append(w)
            pos.append(p)

word_pos_df = pd.DataFrame({'words': words, 'pos': pos})

In [ ]:
word_pos_df[word_pos_df['words'] == ''].drop_duplicates()

In [ ]:
word_pos_df.drop('pos', axis = 1, inplace = True)

In [ ]:
df = pd.DataFrame(word_pos_df['words'].value_counts())

In [ ]:
df = df.reset_index()
df.columns = ['word', 'count']

Computing Frequency

In [ ]:
df['freq'] = df['count']/sum(df['count'])

In [ ]:
df['count'].plot.hist()

Removing Words with Length < 2

In [ ]:
df = df[df['word'].apply(len) > 2]

Removing Words that are used more than 100 times

In [ ]:
df = df[df['count'] < 100]

In [ ]:
df['count'].plot.hist()

Removing Words that are used more than 10 times

In [ ]:
df = df[df['count'] < 10]

In [ ]:
df[df['count'] == 9]['word'].values

In [ ]:
df.shape[0]

Most Frequent 5K words

In [ ]:
fivek_words = pd.read_csv('/content/drive/My Drive/Colab Notebooks/5000_words.csv')

In [ ]:
fivek_words.head()

In [ ]:
df = df[~df['word'].apply(lambda x: x in list(fivek_words['Word'].values))]

In [ ]:
df.shape[0]

In [ ]:
df['word'].tail(20).values

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from IPython.display import clear_output

word_definitions = []
missed = []

for i, w in enumerate(df['word']):
    m = dictionary.meaning(w)
    word_definitions.append((w, m))
    if i%10 == 0:
        clear_output()
        print("{} words completed".format(i))

In [ ]:
len(word_definitions)

In [ ]:
pickle_out = open("word_definitions.pickle", "wb")
pickle.dump(word_definitions, pickle_out)
pickle_out.close()

In [ ]:
word = []
pos = []
meaning = []

for w, m in word_definitions:
    if m is not None:
        for k, v in m.items():
            word.append(w)
            pos.append(k)
            meaning.append(v)

In [ ]:
pd.DataFrame({'word': word, 'pos': pos, 'meaning': meaning})

### Analytics

#### Fetching Data

In [ ]:
def churn_name(book, book_name):

    cleaned_book = []

    for i, text in enumerate(book):

        chap_num_text = inflect.engine().number_to_words(i).capitalize()
        text = text.replace("\n\n"+book_name+" - Chapter {}\n\n\n  \n\nChapter {}\n".format(chap_num_text, chap_num_text), "")
        text = text.replace("\n\n"+book_name+"\n\n\n\n \n\n", "")
        text = text.replace("\n\n"+book_name+" - Epilogue\n\n\n  \n\nEpilogue\n", "")
        cleaned_book.append(text)

    return cleaned_book

In [ ]:
pickle_in = open("/content/drive/My Drive/HHGTG/dont_panic.pickle", "rb")
dont_panic = pickle.load(pickle_in)
pickle_in.close()

In [ ]:
[hhgtg, rateou, luae, slatfat, mh] = dont_panic

In [ ]:
# to open/create a new html file in the write mode
f = open('/content/drive/My Drive/HHGTG/ch1.html', 'w')

# # the html code which will go in the file GFG.html
# html_template = """<html>
# <head>
# <title>Title</title>
# </head>
# <body>
# <h2>Welcome To GFG</h2>

# <p>Default code has been loaded into the Editor.</p>

# </body>
# </html>
# """

# # writing the code into the file
# f.write(html_template)

f.write(hhgtg[0])

# close the file
f.close()



In [ ]:
!pip install fpdf

In [ ]:
len(hhgtg)

In [ ]:
for i in range(len(mh)):
    with open("/content/drive/MyDrive/HHGTG/mh/txt/ch"+str(i+1)+".txt", "w") as text_file:
        text_file.write(mh[i])

In [ ]:
len(mh)

In [ ]:
for i in range(len(slatfat)):
    with open("/content/drive/MyDrive/HHGTG/slatfat/txt/ch"+str(i+1)+".txt", "w") as text_file:
        text_file.write(slatfat[i])

In [ ]:
# Python program to convert
# text file to pdf file


from fpdf import FPDF

for i in range(len(mh)):

    # save FPDF() class into
    # a variable pdf
    pdf = FPDF()

    # Add a page
    pdf.add_page()

    # set style and size of font
    # that you want in the pdf
    pdf.set_font("Arial", size = 15)

    # open the text file in read mode
    f = open("/content/drive/MyDrive/HHGTG/mh/txt/ch"+str(i+1)+".txt", "r")

    # insert the texts in pdf
    for x in f:
        pdf.cell(200, 10, txt = x, ln = 1, align = 'L')

    # save the pdf with name .pdf
    pdf.output("/content/drive/MyDrive/HHGTG/mh/pdf/ch"+str(i+1)+".pdf")



In [ ]:
print(hhgtg[0])

In [ ]:
hhgtg = churn_name(hhgtg, "The Hitch Hiker's Guide to the Galaxy")
rateou = churn_name(rateou, "The Restaurant at the End of the Universe")
luae = churn_name(luae, "Life, the Universe, and Everything")
slatfat = churn_name(slatfat, "So long, and Thanks for all the Fish")
mh = churn_name(mh, "Mostly Harmless")

In [ ]:
dont_panic = [hhgtg, rateou, luae, slatfat, mh]

#### Combining All the Books and Chapters

In [ ]:
dont_panic = ['\n'.join(book) for book in dont_panic]

#### Handling \n

In [ ]:
books = [book.replace("``", "''").split('\n\n\n') for book in dont_panic]

In [ ]:
sentences = [s.replace('\n', ' ') for sentences in books for s in sentences]

In [ ]:
sentences = [s.replace("`", "'") for s in sentences]

#### Fixing double apostrophy

In [ ]:
sentences = [s.replace("''", "\"") for s in sentences]

In [ ]:
sentences = [s.strip() for s in sentences]

#### Removing Empty Strings

In [ ]:
sentences = [s for s in sentences if len(s) > 1]

#### Removing Punctuation and Lemmatizing

In [ ]:
cleaned_sentences = []

for s in sentences:
    cleaned_sentences.append(''.join([c.lower() for c in s if (c in string.ascii_letters) or (c == ' ')]))

In [ ]:
cleaned_sentences = [s.strip() for s in cleaned_sentences]

In [ ]:
cleaned_sentences = [s for s in cleaned_sentences if len(s) > 1]

In [ ]:
cleaned_word_tokens = [word_tokenize(s) for s in cleaned_sentences]

In [ ]:
lem_word_tokens = []

for l in cleaned_word_tokens:
    lem_word_tokens.append([lemmatizer.lemmatize(w) for w in l])

#### Stemming

In [ ]:
stem_word_tokens = []

stop_words_no_punc = [word.translate(str.maketrans('', '', string.punctuation)) for word in stop_words]

for l in lem_word_tokens:
    stem_word_tokens.append([ps.stem(w) for w in l if w not in stop_words_no_punc])

# len(set(list(pd.Series(cleaned_word_tokens).explode())))
# len(set(list(pd.Series(lem_word_tokens).explode())))
# len(set(list(pd.Series(stem_word_tokens).explode())))

#### Creating Index Dictionaries

In [ ]:
idx_to_word = {i: w for i, w in enumerate(sorted(set(list(pd.Series(cleaned_word_tokens).explode()))))}
word_to_idx = {w: i for i, w in enumerate(sorted(set(list(pd.Series(cleaned_word_tokens).explode()))))}

#### Manual Term Frequency (Unused Code)

In [ ]:
df = pd.Series(cleaned_word_tokens)

In [ ]:
# for padding
idx_to_word[len(idx_to_word)] = ''
word_to_idx[''] = len(idx_to_word)

In [ ]:
df = df.apply(lambda x: [word_to_idx[t] for t in x])

In [ ]:
df = df[df.apply(len) > 2]
df = df[df.apply(len) < 150]

In [ ]:
# max_len = max(df.apply(len))
# max_len
# df = df.apply(lambda x: x + [0]*(max_len - len(x)))

In [ ]:
def one_hot(x, classes = len(word_to_idx)):
    a = np.zeros(classes)
    a[np.array(x)] = 1
    return a

In [ ]:
df = df.apply(one_hot)

In [ ]:
hot_bag = np.array([i for i in df])

In [ ]:
%%time
dot_doc = np.dot(hot_bag, hot_bag.T)

In [ ]:
%%time
dot_term = np.dot(hot_bag.T, hot_bag)

#### Term Frequency Matrix

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
df = pd.DataFrame([' '.join(i) for i in stem_word_tokens], columns=['sentences'])

vocab_list = list(sorted(set([w for s in stem_word_tokens for w in s if ((len(w) > 2) and (len(w) < 12))])))

vectorizer = CountVectorizer(vocabulary = vocab_list, tokenizer=word_tokenize)

X = vectorizer.fit_transform(df['sentences'].values)

In [ ]:
tf_df = pd.DataFrame(data=X.toarray(), columns=vectorizer.get_feature_names())

In [ ]:
tf_df[(tf_df.sum(axis = 1) > 2) & (tf_df.sum(axis = 1) < 150)].sum(axis = 1).plot.hist()

In [ ]:
tf_df = tf_df[(tf_df.sum(axis = 1) > 2) & (tf_df.sum(axis = 1) < 50)]

In [ ]:
tf = tf_df.values

In [ ]:
np.random.shuffle(tf)

In [ ]:
tf.shape

#### K Means Clustering

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
%%time
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters = i, random_state = 42, n_jobs = -1)
    kmeans.fit(tf)
    wcss.append(kmeans.inertia_)

In [ ]:
plt.plot(range(1, 11), wcss)

In [ ]:
kmeans = KMeans(n_clusters = 4, random_state = 42, n_jobs = -1)

In [ ]:
%%time
labls = kmeans.fit_predict(tf)

In [ ]:
tf_df['cluster_num'] = labls

In [ ]:
cluster_df = pd.DataFrame()
for i in set(kmeans.labels_):
    if cluster_df.empty:
        cluster_df = pd.DataFrame(tf_df[tf_df['cluster_num'] == i].sum(axis = 0)).rename(columns = {0: 'cluster_'+str(i)})
    else:
        cluster_df = cluster_df.join(pd.DataFrame(tf_df[tf_df['cluster_num'] == i].sum(axis = 0)).rename(columns = {0: 'cluster_'+str(i)}))

###### Clusters

In [ ]:
cluster_df.drop('cluster_num', inplace = True)

In [ ]:
cluster_df

In [ ]:
c1 = cluster_df[cluster_df['cluster_0'] > 0]['cluster_0']
c1.sort_values(ascending = False)

In [ ]:
cluster_df[cluster_df['cluster_1'] > 0]['cluster_1'].sort_values(ascending = False)

In [ ]:
cluster_df[cluster_df['cluster_2'] > 0]['cluster_2'].sort_values(ascending = False)

In [ ]:
cluster_df[cluster_df['cluster_3'] > 0]['cluster_3'].sort_values(ascending = False)

In [ ]:
cluster_df[cluster_df['cluster_4'] > 0]['cluster_4'].sort_values(ascending = False)

In [ ]:
cluster_df[cluster_df['cluster_5'] > 0]['cluster_5'].sort_values(ascending = False)

In [ ]:
cluster_df[cluster_df['cluster_6'] > 0]['cluster_6'].sort_values(ascending = False)

In [ ]:
cluster_df[cluster_df['cluster_6'] > 0]['cluster_6'].sort_values(ascending = False)

In [ ]:
df = df.loc[tf_df.index]
df['cluster_num'] = kmeans.labels_

In [ ]:
df['cluster_num'].value_counts()

In [ ]:
df[df['cluster_num'] == 1]['sentences']

### TFIDF

In [ ]:
stem_words_df = pd.Series(stem_word_tokens)
stem_words_df.apply(len).sort_values().plot.hist()

In [ ]:
stem_words_df = stem_words_df[stem_words_df.apply(len) < 60].apply(lambda x: ' '.join(x))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [ ]:
vectorizer = TfidfVectorizer(stop_words=ENGLISH_STOP_WORDS)

In [ ]:
X = vectorizer.fit_transform(stem_words_df)

In [ ]:
tf_idf = pd.DataFrame(X.toarray(), columns = vectorizer.get_feature_names())

In [ ]:
tf_idf = tf_idf[[i for i in vectorizer.get_feature_names() if (len(i) > 2) & (len(i) < 12)]]

In [ ]:
tf_idf = tf_idf.loc[tf_idf.sum(axis = 1) != 0.0]

In [ ]:
tf_idf.sum().max()

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
%%time
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters = i, random_state = 42, n_jobs = -1)
    kmeans.fit(tf_idf)
    wcss.append(kmeans.inertia_)

In [ ]:
plt.plot(range(1, 11), wcss)

In [ ]:
kmeans = KMeans(n_clusters = 4, random_state = 42, n_jobs = -1)

In [ ]:
%%time
labls = kmeans.fit_predict(tf_idf)

In [ ]:
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

In [ ]:
for i in range(4):
    print('Cluster %d:' % i),
    for t in order_centroids[i, :10]:
        print(vectorizer.get_feature_names()[t])
    print('\n')